# SpatialGDC on BRCA
Breast cancer spatial transcriptomics (Visium, 21-class fine annotation).

In [ ]:
import scanpy as sc, pandas as pd, numpy as np, sys
sys.path.insert(0, "..")
from spatialgdc import SpatialGDC, prepare_graph, compute_spatial_keep_prob, clustering, fix_seed
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

fix_seed(0)

## 1. Load and Preprocess

In [ ]:
brca_dir = "../dataset/BRCA1/V1_Human_Breast_Cancer_Block_A_Section_1"
adata = sc.read_visium(brca_dir)
adata.var_names_make_unique()
adata.obs_names = adata.obs_names.str.replace("-1", "", regex=False).str.strip()

# Load fine-grained labels (21-class)
meta = pd.read_csv("../dataset/BRCA1/metadata.tsv", sep="\t", index_col=0)
meta.index = meta.index.astype(str).str.replace("-1", "", regex=False).str.strip()
adata.obs["Region"] = meta.loc[adata.obs_names, "fine_annot_type"]
n_clusters = adata.obs["Region"].nunique()

# Preprocessing with HVG filtering
sc.pp.filter_genes(adata, min_counts=1)
sc.pp.filter_cells(adata, min_counts=1)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=3000)
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata)
adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X.toarray())

print(f"Spots: {adata.shape[0]}, HVGs: {adata.shape[1]}, Clusters: {n_clusters}")

## 2. Build Graphs and Train

In [ ]:
g_spatial = prepare_graph(adata, "spatial")
g_expr = prepare_graph(adata, "expr")

coords = adata.obsm["spatial"].copy()
coords = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
keep_prob = compute_spatial_keep_prob(g_expr, coords, sigma=0.6)

model = SpatialGDC(
    input_data=adata.obsm["X_pca"].copy(),
    graph_dict={"spatial": g_spatial, "expr": g_expr},
    n_clusters=n_clusters,
    expr_keep_prob=keep_prob,
    gamma=5.0, kappa=0.05, beta=2.0,
    use_spatial_drop=True, use_intersection_cl=True,
)
pred_labels, embeddings, x_rec = model.train()

## 3. Evaluate

In [ ]:
adata.obsm["emb"] = embeddings
clustering(adata, n_clusters, key="emb", refinement=True, cluster_methods="mclust")

cc = "mclust_refined" if "mclust_refined" in adata.obs.columns else "mclust"
ae = adata[adata.obs.Region.notna()]
ari = adjusted_rand_score(ae.obs["Region"], ae.obs[cc])
print(f"ARI = {ari:.4f}")

## 4. Visualize

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.spatial(adata, color="Region", ax=axes[0], show=False, title="Ground Truth (21 classes)", legend_loc=None)
sc.pl.spatial(adata, color=cc, ax=axes[1], show=False, title=f"SpatialGDC (ARI={ari:.3f})", legend_loc=None)
plt.tight_layout(); plt.show()